# BCDR replication validation

Runs on the **Microsoft Sentinel data lake** (VS Code Sentinel extension, PySpark kernel).

It compares **yesterday's full 24h** event counts, per table, between:

1. the **primary Sentinel workspace** (the source of the logs), and
2. the **federated Delta tables** produced by the BCDR Azure Data Factory pipeline.

For every table the counts must match. Each table is written as one row to an
analytics-tier custom table (`ReplicationValidation_CL`) with a `Mismatch` flag, so a
Microsoft Sentinel analytic rule can raise an Informational alert and a playbook can email
the SOC team.

Schedule this notebook as a **Daily** job (see README.md).

## 1. Parameters

Edit these to match your environment. Everything else is derived from them.

In [ ]:
# Name of the PRIMARY Sentinel workspace (the data-lake database that holds the source logs).
# Run data_provider.list_databases() in a scratch cell if you are unsure of the exact name.
PRIMARY_WORKSPACE_NAME = "bemorene"

# Workspace (analytics tier) where the ReplicationValidation_CL result table is created/appended.
# Usually the same Sentinel workspace.
ANALYTICS_TIER_WORKSPACE_NAME = "bemorene"

# Name of the data-lake federation connector instance that exposes the BCDR Delta tables.
# Federated tables are named '<table>_<FEDERATION_INSTANCE_NAME>'. Setting this explicitly is
# strongly recommended. Leave "" to auto-detect (best effort).
FEDERATION_INSTANCE_NAME = ""

# Analytics-tier result table. The '_SPRK_CL' suffix tells the provider to write to the
# analytics tier; it surfaces in Log Analytics / Sentinel as 'ReplicationValidation_CL'.
RESULT_TABLE_NAME = "ReplicationValidation_SPRK_CL"

# A table is flagged Mismatch=True when its gap percentage is strictly greater than this.
# 0.0 means counts must match exactly. Raise it (e.g. 1.0) to tolerate small in-flight gaps.
GAP_TOLERANCE_PERCENT = 0.0

# How many full days back to validate. 1 = yesterday (00:00 UTC to today 00:00 UTC).
LOOKBACK_DAYS = 1

# Optional: restrict to specific source tables (case-insensitive). Empty = all replicated tables.
TABLE_ALLOWLIST = []

# Event-time column used on both sides so the comparison is apples-to-apples.
TIME_COLUMN = "TimeGenerated"

## 2. Initialize the provider and compute the validation window

In [ ]:
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType,
)
from sentinel_lake.providers import MicrosoftSentinelProvider

data_provider = MicrosoftSentinelProvider(spark)

# Yesterday's full 24h in UTC: [today 00:00 - LOOKBACK_DAYS, today 00:00).
now_utc = datetime.now(timezone.utc)
window_end = datetime(now_utc.year, now_utc.month, now_utc.day, tzinfo=timezone.utc)
window_start = window_end - timedelta(days=LOOKBACK_DAYS)

window_start_str = window_start.strftime("%Y-%m-%d %H:%M:%S")
window_end_str = window_end.strftime("%Y-%m-%d %H:%M:%S")
print(f"Validation window (UTC): {window_start_str}  ->  {window_end_str}")

## 3. Discover tables and map primary <-> federated

Federated Delta tables live under the **System tables** database and are named
`<table>_<FEDERATION_INSTANCE_NAME>`. We match them to primary-workspace tables by
case-insensitive base name.

In [ ]:
primary_tables = data_provider.list_tables(PRIMARY_WORKSPACE_NAME)
system_tables = data_provider.list_tables()  # System tables db includes federated tables


def build_federated_map(system_tables, instance):
    """Return {normalized_base_name: federated_table_name}."""
    fed_map = {}
    if instance:
        suffix = "_" + instance.lower()
        for t in system_tables:
            if t.lower().endswith(suffix):
                base = t[: -len(suffix)]
                fed_map[base.lower()] = t
    else:
        # Best-effort auto-detect: treat the text before the last underscore as the base name.
        for t in system_tables:
            if "_" in t:
                base = t.rsplit("_", 1)[0]
                fed_map.setdefault(base.lower(), t)
    return fed_map


fed_map = build_federated_map(system_tables, FEDERATION_INSTANCE_NAME)
allow = {a.lower() for a in TABLE_ALLOWLIST}

tables_to_check = []
for ptable in sorted(primary_tables):
    base = ptable.lower()
    if allow and base not in allow:
        continue
    if base in fed_map:
        tables_to_check.append((ptable, fed_map[base]))

print(f"Primary tables: {len(primary_tables)} | federated tables matched: {len(fed_map)}")
print(f"Tables to validate: {len(tables_to_check)}")
for p, f_ in tables_to_check:
    print(f"  {p}  <->  {f_}")

## 4. Count events per table and build the result set

In [ ]:
def count_events(table_name, database_name=None):
    """Count rows whose event time falls inside the validation window. None on read error."""
    try:
        df = (
            data_provider.read_table(table_name, database_name)
            if database_name
            else data_provider.read_table(table_name)
        )
        if TIME_COLUMN not in df.columns:
            print(f"  ! {table_name}: no '{TIME_COLUMN}' column, skipping")
            return None
        df = df.filter(
            (F.col(TIME_COLUMN) >= F.lit(window_start_str).cast("timestamp"))
            & (F.col(TIME_COLUMN) < F.lit(window_end_str).cast("timestamp"))
        )
        return df.count()
    except Exception as exc:  # noqa: BLE001 - surface the table that failed and continue
        print(f"  ! could not read {table_name}: {exc}")
        return None


results = []
for ptable, ftable in tables_to_check:
    primary_count = count_events(ptable, PRIMARY_WORKSPACE_NAME)
    parquet_count = count_events(ftable)
    if primary_count is None or parquet_count is None:
        continue
    if primary_count == 0:
        gap = 0.0 if parquet_count == 0 else 100.0
    else:
        gap = abs(primary_count - parquet_count) / primary_count * 100.0
    mismatch = gap > GAP_TOLERANCE_PERCENT
    results.append((ptable, int(primary_count), int(parquet_count), round(gap, 2), bool(mismatch)))
    print(f"  {ptable}: primary={primary_count} parquet={parquet_count} gap={round(gap, 2)}% mismatch={mismatch}")

In [ ]:
schema = StructType(
    [
        StructField("Table", StringType(), False),
        StructField("NumEventsInPrimaryWorkspace", LongType(), False),
        StructField("NumEventsInParquetFiles", LongType(), False),
        StructField("GapPercentage", DoubleType(), False),
        StructField("Mismatch", BooleanType(), False),
    ]
)

results_df = spark.createDataFrame(results, schema)
results_df = (
    results_df.withColumn("TimeGenerated", F.current_timestamp())
    .withColumn("WindowStartUtc", F.lit(window_start_str).cast("timestamp"))
    .withColumn("WindowEndUtc", F.lit(window_end_str).cast("timestamp"))
)
results_df.orderBy(F.col("Mismatch").desc(), F.col("GapPercentage").desc()).show(200, truncate=False)

## 5. Write the result to the analytics tier

Appends one row per evaluated table to `ReplicationValidation_CL`. The analytics tier is
**append-only**, which is exactly what we want for a daily history. The Sentinel analytic rule
filters this table for `Mismatch == true`.

In [ ]:
row_count = results_df.count()
if row_count == 0:
    print("No tables were evaluated. Nothing written. Check PRIMARY_WORKSPACE_NAME / FEDERATION_INSTANCE_NAME.")
else:
    run_id = data_provider.save_as_table(
        results_df,
        RESULT_TABLE_NAME,
        ANALYTICS_TIER_WORKSPACE_NAME,
        write_options={"mode": "append"},
    )
    print(f"Wrote {row_count} rows to {RESULT_TABLE_NAME}. Run ID: {run_id}")

## 6. Summary

In [ ]:
mismatches = [r for r in results if r[4]]
print(f"Validation window (UTC): {window_start_str} -> {window_end_str}")
print(f"Tables evaluated: {len(results)}")
print(f"Mismatches: {len(mismatches)}")
for m in mismatches:
    print(f"  MISMATCH  {m[0]}: primary={m[1]} parquet={m[2]} gap={m[3]}%")
if not mismatches and results:
    print("All replicated tables match. \u2713")